In [ ]:
!pip install -q streamlit shap pyngrok


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.7 MB/s eta 0:00:00


In [ ]:
!pkill streamlit
!pkill ngrok

In [ ]:
!ngrok config add-authtoken 2vZ4QdExP51XqLLyuzYitSJRRsJ_6XvZDd6FMnpUockpuvUp2

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
app_code = '''
import streamlit as st
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import shap
import pandas as pd
import numpy as np

st.set_page_config(layout="wide")
st.title("🧠 Explainable Depression Detection (ICD-9: 311)")

model_path = "/content/icd9"  # upload a folder named 'icd9' into Colab with model files

model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model.eval()

user_input = st.text_area("Paste a discharge summary below:", height=300)

if st.button("Predict and Explain"):
    if not user_input.strip():
        st.warning("Please enter some text.")
    else:
        #Tokenize for classification
        inputs = tokenizer(user_input, return_tensors="pt", truncation=True, padding="max_length", max_length=512)
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0].detach().numpy()
        depression_prob = probs[1] * 100
        not_depression_prob = probs[0] * 100

        if depression_prob >= 50:
            st.subheader("✅ Depression Detected (ICD-9: 311)")
        else:
            st.subheader("❌ No Depression Detected")

        st.write(f"Confidence: {depression_prob:.2f}% depression | {not_depression_prob:.2f}% no depression")


        #SHAP Text Explanation
        st.subheader("🔍 Highlighted Words (SHAP Text Visualization)")

        # SHAP pipeline
        pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, return_all_scores=True)
        explainer = shap.Explainer(pipe)

        #Handle long input for SHAP
        full_token_count = len(tokenizer.encode(user_input))
        if full_token_count > 512:
            st.warning("⚠️ Input was too long and has been truncated to 512 tokens for explanation.")

        encoded = tokenizer(user_input, truncation=True, max_length=512, return_tensors="pt", return_token_type_ids=False)
        truncated_text = tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=True)

        shap_values = explainer([truncated_text])
        tokens = shap_values[0].data
        raw_contributions = shap_values[0].values
        contributions = np.array(raw_contributions).flatten()

        #Convert token IDs to text tokens if needed
        if isinstance(tokens[0], (int, np.integer)):
            tokens = tokenizer.convert_ids_to_tokens(tokens)

        #Clean tokens
        clean_tokens = [t.replace("▁", " ").replace("##", "") for t in tokens]

        #Top 10 Most Impactful Tokens Table
        filtered = [(t, s) for t, s in zip(clean_tokens, contributions) if t.isalpha() and len(t) > 2]
        if not filtered:
            st.warning("⚠️ No meaningful token contributions could be extracted.")
        else:
            top_contributors = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)[:10]
            top_table = pd.DataFrame(top_contributors, columns=["Token", "SHAP Contribution"])
            st.subheader("📊 Top 10 Most Influential Words")
            st.table(top_table)

        #Highlighted Token Heatmap View
        st.subheader("🎨 Highlighted Words (Heatmap View)")
        max_contribution = np.max(np.abs(contributions)) if np.max(np.abs(contributions)) > 0 else 1e-6

        html = ""
        for token, score in zip(clean_tokens, contributions):
            alpha = min(0.8, abs(score) / max_contribution + 0.05)
            color = f"rgba(255,0,0,{alpha})" if score > 0 else f"rgba(0,0,255,{alpha})"
            html += f'<span style="background-color:{color}; padding:1px 2px; margin:1px; border-radius:4px">{token}</span> '

        if np.allclose(contributions, 0, atol=1e-5):
            st.info("ℹ️ The model found very weak token-level contributions. Consider revising input.")

        st.markdown(
            f'<div style="max-height: 400px; overflow-y: auto; line-height: 1.6;">{html}</div>',
            unsafe_allow_html=True
        )

        #Legend
        st.markdown(
            "<p style='margin-top:10px; font-size: 0.9em;'>"
            "🔴 = pushes model toward <b>No Depression</b> &nbsp;&nbsp; | &nbsp;&nbsp;"
            "🔵 = pushes model toward <b>Depression</b>"
            "</p>",
            unsafe_allow_html=True
        )
'''
#write the code to app.py so we can run it
with open("app.py", "w") as f:
    f.write(app_code)


In [ ]:
import subprocess
import time
from pyngrok import ngrok

#Kill the app if already running
!pkill streamlit

#Start Streamlit using subprocess, running our app.py
process = subprocess.Popen(["streamlit", "run", "app.py"])

#Open ngrok tunnel for public access
public_url = ngrok.connect(8501, "http")
print("Streamlit app is live at:", public_url)


🚀 Streamlit app is live at: NgrokTunnel: "https://ca46-34-16-161-110.ngrok-free.app" -> "http://localhost:8501"
